# Data Preparation 
- Scraping 100 cities data across different tags to find the accessibility and mobility.
- Using `GEOAPIFY` to extract data.

## Data Extraction


In [1]:
# Libraries importing
import requests
import time
import numpy as np
import pandas as pd

In [2]:
# GEOAPIFY initialization
import os
from dotenv import load_dotenv
load_dotenv()
API_KEY =  os.getenv("GEOAPIFY_KEY")

if not API_KEY:
    raise ValueError("GEOAPIFY_API_KEY not found — check your .env file")

GEOCODE_URL = "https://api.geoapify.com/v1/geocode/search"
PLACES_URL = "https://api.geoapify.com/v2/places"


In [3]:
#WIKIDATA Initialization
WIKIDATA_URL = "https://query.wikidata.org/sparql"
WIKIDATA_HEADERS = {"User-Agent": "CityWalkMobilityIQ/1.0"}

In [4]:
CITIES = [
    "Hong Kong, Hong Kong",
    "Bangkok, Thailand",
    "London, United Kingdom",
    "Macau, Macau",
    "Singapore, Singapore",
    "Paris, France",
    "Dubai, United Arab Emirates",
    "New York City, United States",
    "Kuala Lumpur, Malaysia",
    "Istanbul, Turkey",
    "Delhi, India",
    "Antalya, Turkey",
    "Los Angeles, United States",
    "Shenzhen, China",
    "Mumbai, India",
    "Phuket, Thailand",
    "Rome, Italy",
    "Tokyo, Japan",
    "Pattaya, Thailand",
    "Taipei, Taiwan",
    "Mecca, Saudi Arabia",
    "Guangzhou, China",
    "Prague, Czech Republic",
    "Medina, Saudi Arabia",
    "Seoul, South Korea",
    "Amsterdam, Netherlands",
    "Agra, India",
    "Miami, United States",
    "Osaka, Japan",
    "Las Vegas, United States",
    "Shanghai, China",
    "Ho Chi Minh City, Vietnam",
    "Surabaya, Indonesia",          
    "Barcelona, Spain",
    "Cairo, Egypt",
    "Milan, Italy",
    "Chennai, India",
    "Vienna, Austria",
    "Johor Bahru, Malaysia",
    "Jaipur, India",
    "Cancun, Mexico",
    "Berlin, Germany",
    "Athens, Greece",
    "Orlando, United States",
    "Moscow, Russia",
    "Venice, Italy",
    "Madrid, Spain",
    "Ha Long, Vietnam",
    "Riyadh, Saudi Arabia",
    "Dublin, Ireland",
    "Florence, Italy",
    "Jerusalem, Israel",
    "Hanoi, Vietnam",
    "Toronto, Canada",
    "Johannesburg, South Africa",
    "Sydney, Australia",
    "Munich, Germany",
    "Jakarta, Indonesia",
    "Beijing, China",
    "Saint Petersburg, Russia",
    "Brussels, Belgium",
    "Budapest, Hungary",
    "Lima, Peru",
    "Lisbon, Portugal",
    "Dammam, Saudi Arabia",
    "Penang, Malaysia",
    "Heraklion, Greece",
    "Kyoto, Japan",
    "Zhuhai, China",
    "Vancouver, Canada",
    "Chiang Mai, Thailand",
    "Copenhagen, Denmark",
    "San Francisco, United States",
    "Melbourne, Australia",
    "Warsaw, Poland",
    "Marrakesh, Morocco",
    "Kolkata, India",
    "Cebu City, Philippines",
    "Auckland, New Zealand",
    "Tel Aviv, Israel",
    "Guilin, China",
    "Honolulu, United States",
    "Hurghada, Egypt",
    "Krakow, Poland",
    "Zurich, Switzerland",
    "Buenos Aires, Argentina",
    "Chiba, Japan",
    "Frankfurt, Germany",
    "Stockholm, Sweden",
    "Da Nang, Vietnam",
    "Denpasar, Indonesia",           
    "Nice, France",
    "Fukuoka, Japan",
    "Abu Dhabi, United Arab Emirates",
    "Jeju, South Korea",
    "Porto, Portugal",
    "Rhodes, Greece",
    "Rio de Janeiro, Brazil",
    "Krabi, Thailand",
    "Bangalore, India",
]

print(f"Total cities: {len(CITIES)}")

Total cities: 100


In [5]:
def safe_get(url, params, headers=None, retries=3, backoff_seconds=5):
    """
    Makes a GET request, and if it fails (e.g. network hiccup), waits
    a few seconds and tries again — up to 'retries' times.
    """
    for attempt_number in range(retries):
        try:
            return requests.get(url, params=params, headers=headers, timeout=45)
        except requests.exceptions.RequestException as error:
            print(f"    Attempt {attempt_number + 1} failed: {error}")
            is_last_attempt = (attempt_number == retries - 1)
            if not is_last_attempt:
                time.sleep(backoff_seconds)
    return None

In [6]:
def get_population_by_coordinates(lat, lon, radius_km=10, min_population=20_000):
    """
    Finds the population of a city near given coordinates. Requires at
    least min_population to avoid false matches to tiny nearby places —
    20,000 was chosen since it comfortably clears the largest false
    match found during testing (2,802) while still accepting genuinely
    small cities in the dataset (e.g. Mugla, Turkey at 64,706).
    """
    query = f"""
    SELECT ?city ?cityLabel ?population WHERE {{
        SERVICE wikibase:around {{
            ?city wdt:P625 ?location.
            bd:serviceParam wikibase:center "Point({lon} {lat})"^^geo:wktLiteral.
            bd:serviceParam wikibase:radius "{radius_km}".
        }}
        ?city wdt:P31/wdt:P279* wd:Q515.
        ?city wdt:P1082 ?population.
        SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    ORDER BY DESC(?population)
    LIMIT 1
    """
    response = safe_get(WIKIDATA_URL, params={"query": query, "format": "json"}, headers=WIKIDATA_HEADERS)
    if response is None or response.status_code != 200:
        return None
    try:
        data = response.json()
    except ValueError:
        return None

    results = data.get("results", {}).get("bindings", [])
    if results:
        population = int(results[0]["population"]["value"])
        if population >= min_population:
            return population
    return None

In [7]:
def get_radius_safe(population: int) -> int:
    """
    Picks a search radius based on city population, capped 8km. Handle by Geopaify API reliably.
    """
    population_scale = (population / 1_000_000) ** 0.5
    radius_km = 5 * population_scale
    radius_km_capped = min(max(radius_km, 5), 8)
    return int(radius_km_capped * 1000)  # convert to meters

In [8]:
def get_places_count(lon, lat, radius_m, cat_string, max_pages=3):
    """
    Counts how many places of a given category (e.g. restaurants) exist
    near a location. Geoapify only returns 500 results per request, so pagination added to search more than 500 results per request.
    """
    total_count = 0

    for page_number in range(max_pages):
        params = {
            "categories": cat_string,
            "filter": f"circle:{lon},{lat},{radius_m}",
            "limit": 500,
            "offset": page_number * 500,
            "apiKey": API_KEY,
        }
        response = safe_get(PLACES_URL, params)
        if response is None:
            return None

        results_this_page = len(response.json().get("features", []))
        total_count += results_this_page
        time.sleep(0.5)  # avoid hitting Geoapify's rate limit

        if results_this_page < 500:
            break  # no more pages left

    return total_count

In [9]:
def get_places_features(lon, lat, radius_m, cat_string, max_pages=3):
    """
    Same idea as get_places_count, but returns the FULL place data
    (not just a count).
    """
    all_places = []

    for page_number in range(max_pages):
        params = {
            "categories": cat_string,
            "filter": f"circle:{lon},{lat},{radius_m}",
            "limit": 500,
            "offset": page_number * 500,
            "details": "details.accessibility",
            "apiKey": API_KEY,
        }
        response = safe_get(PLACES_URL, params)
        if response is None:
            break

        places_this_page = response.json().get("features", [])
        all_places.extend(places_this_page)
        time.sleep(0.5)

        if len(places_this_page) < 500:
            break

    return all_places

In [10]:
def get_raw_tag(place, tag_name):
    """
    Safely pulls a specific OSM tag. Returns None if the tag doesn't exist.
    """
    properties = place.get("properties", {})
    datasource = properties.get("datasource", {})
    raw_tags = datasource.get("raw", {})
    return raw_tags.get(tag_name)

In [11]:
def count_wheelchair_accessible(places):
    """
    Given a list of places (e.g. restaurants), counts how many have a
    wheelchair tag at all, and how many of those are actually accessible.
    Returns (number_tagged, number_accessible).
    """
    number_tagged = 0
    number_accessible = 0

    for place in places:
        wheelchair_value = get_raw_tag(place, "wheelchair")
        if wheelchair_value is not None:
            number_tagged += 1
            if wheelchair_value in ["yes", "limited", "designated"]:
                number_accessible += 1

    return number_tagged, number_accessible

In [12]:
def get_poi_accessibility_rate(places):
    """
    Turns a list of places into a single 'accessibility rate' percentage:
    of the places that HAVE a wheelchair tag, what % are accessible?
    Places with no tag at all are excluded.
    """
    number_tagged, number_accessible = count_wheelchair_accessible(places)
    if number_tagged == 0:
        return 0.0
    return round((number_accessible / number_tagged) * 100, 2)

In [13]:
def get_street_walkability(street_places):
    """
    Looks at footway/street data and computes two simple percentages:
    - what % of streets are paved (vs dirt/gravel/etc)
    - what % of streets are rated 'good' or 'excellent' smoothness
    """
    total_streets = len(street_places)
    if total_streets == 0:
        return {"pct_paved_streets": None, "pct_good_smoothness": None}

    paved_surfaces = ["paved", "asphalt", "concrete", "paving_stones"]
    good_smoothness_ratings = ["excellent", "good"]

    paved_count = sum(1 for p in street_places if get_raw_tag(p, "surface") in paved_surfaces)
    smooth_count = sum(1 for p in street_places if get_raw_tag(p, "smoothness") in good_smoothness_ratings)

    return {
        "pct_paved_streets": round((paved_count / total_streets) * 100, 2),
        "pct_good_smoothness": round((smooth_count / total_streets) * 100, 2),
    }

In [14]:
def get_city_features(city_name: str, radius_m: int = None) -> dict:
    """
    Main function: given a city name, fetches its location, then builds
    a dictionary of features describing how walkable/accessible that
    city is. Population is looked up by COORDINATES (not by name), since
    Geoapify and Wikidata often use different names for the same place —
    matching by location sidesteps that mismatch. Radius is computed
    from population if not explicitly provided.
    """
    geo_params = {"text": city_name, "type": "city", "apiKey": API_KEY}
    geo_response = safe_get(GEOCODE_URL, params=geo_params)

    if not geo_response or not geo_response.json().get("features"):
        return None

    city_location = geo_response.json()["features"][0]["properties"]
    lon = city_location["lon"]
    lat = city_location["lat"]

    population = get_population_by_coordinates(lat, lon)
    if population is None:
        print(f"    Warning: no population found for {city_name} — skipping")
        return None

    if radius_m is None:
        radius_m = get_radius_safe(population)

    accessibility_checked_categories = {
        "dining": "catering.restaurant",
        "hotel": "accommodation.hotel",
        "attraction": "tourism.attraction",
    }
    density_only_categories = {
        "healthcare": "healthcare",
        "essential": "commercial.supermarket,commercial.convenience",
        "transit": "public_transport",
        "bench": "leisure.park",
        "toilet": "building.toilet",
        "parking": "parking",
    }

    city_data = {"city": city_name, "population": population}
    area_km2 = np.pi * (radius_m / 1000) ** 2

    total_places_checked = 0
    total_places_tagged = 0

    for category_name, category_string in accessibility_checked_categories.items():
        places = get_places_features(lon, lat, radius_m, category_string)
        total_places_checked += len(places)
        number_tagged, _ = count_wheelchair_accessible(places)
        total_places_tagged += number_tagged
        city_data[f"{category_name}_access_rate"] = get_poi_accessibility_rate(places)

    for category_name, category_string in density_only_categories.items():
        count = get_places_count(lon, lat, radius_m, category_string)
        city_data[f"{category_name}_density"] = round(count / area_km2, 2) if count is not None else np.nan

    city_data["tag_coverage_pct"] = round((total_places_tagged / total_places_checked) * 100, 2) if total_places_checked > 0 else 0

    street_places = get_places_features(lon, lat, radius_m, "highway.pedestrian")
    city_data.update(get_street_walkability(street_places))

    return city_data

In [ ]:
output_path = "../data/raw/city_accessibility_raw.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

city_records = []
print("Starting full extraction pipeline...\n")

for city in CITIES:
    try:
        print(f"Processing {city}...")
        features = get_city_features(city)
        if features is not None:
            city_records.append(features)
            pd.DataFrame(city_records).to_csv(output_path, index=False)
        else:
            print(f"  -> skipped, no features")
    except Exception as e:
        print(f"  -> ERROR on {city}: {e} — skipping, batch continues")
        continue
    time.sleep(1)

print(f"\nDone. Collected {len(city_records)} of {len(CITIES)} cities.")

Starting full extraction pipeline...

Processing Hong Kong, Hong Kong...
  -> skipped, no features
Processing Bangkok, Thailand...
Processing London, United Kingdom...
Processing Macau, Macau...
  -> skipped, no features
Processing Singapore, Singapore...
Processing Paris, France...
    Attempt 1 failed: HTTPSConnectionPool(host='api.geoapify.com', port=443): Read timed out. (read timeout=45)
Processing Dubai, United Arab Emirates...
Processing New York City, United States...
Processing Kuala Lumpur, Malaysia...
    Attempt 1 failed: HTTPSConnectionPool(host='query.wikidata.org', port=443): Read timed out. (read timeout=45)
Processing Istanbul, Turkey...
Processing Delhi, India...
Processing Antalya, Turkey...
Processing Los Angeles, United States...
Processing Shenzhen, China...
Processing Mumbai, India...
Processing Phuket, Thailand...
    Attempt 1 failed: HTTPSConnectionPool(host='query.wikidata.org', port=443): Read timed out. (read timeout=45)
    Attempt 2 failed: HTTPSConnectio

#### Observation: City Data Collection
- Collected data for 92 of 100 planned cities (92%).
- 8 cities could not be resolved: Hong Kong, Macau, Mecca, Ha Long,
  Rhodes, and Krabi returned no population match from Wikidata even
  after debugging.
#### Conclusion
The final dataset covers 92 cities across major countries and 6
continents, sufficient for the project's modeling goals.